# Cargo Network Optimization - PuLP Linear Programming

This notebook reproduces the full WGU Task 3 cargo-network linear programming model. The demand centers and lane-cost tables are embedded directly from the original coursework notebook so the result can be regenerated end-to-end.

**Verified result:**
- Solver status: **Optimal**
- Total minimum cost: **$200,863.75**
- CVG utilization: **95,650 / 95,650 tons**
- AFW utilization: **38,097 / 44,350 tons**
- All hub capacity, focus-city capacity, focus-city flow-balance, and demand-satisfaction checks pass


In [1]:
# Imports
from pulp import LpProblem, LpVariable, LpMinimize, LpStatus, lpSum, PULP_CBC_CMD
import pandas as pd

# Defining Data
# Hub cities (name, current tons, capacity)
hubs = {
    "CVG": {"current_tons": 82800, "capacity": 95650},
    "AFW": {"current_tons": 38400, "capacity": 44350}
}

# Focus Cities (name, airport, capacity)
focus_cities = {
    "Leipzig": {"airport": "Leipzig/Halle Airport", "capacity": 85000},
    "Hyderabad": {"airport": "Rajiv Gandhi International Airport", "capacity": 19000},
    "San Bernardino": {"airport": "San Bernardino International Airport", "capacity": 36000}
}

# Demand Centers (country -> city -> demand)
# 
centers = {
    "France": {"Paris": 6500},
    "Germany": {"Cologne": 640, "Hanover": 180},
    "India": {
        "Bengaluru": 9100, "Coimbatore": 570, "Delhi": 19000, "Mumbai": 14800
    },
    "Italy": {
        "Cagliari": 90, "Catania": 185, "Milan": 800, "Rome": 1700
    },
    "Poland": {"Katowice": 170},
    "Spain": {"Barcelona": 2800, "Madrid": 3700},
    "United Kingdom": {"Castle Donington": 30, "London": 6700},
    "United States": {
        "Mobile": 190, "Anchorage": 175, "Fairbanks": 38, "Phoenix": 2400,
        "Los Angeles": 7200, "Ontario": 100, "Riverside": 1200, "Sacramento": 1100,
        "San Francisco": 1900, "Stockton": 240, "Denver": 1500, "Hartford": 540,
        "Miami": 3400, "Lakeland": 185, "Tampa": 1600, "Atlanta": 3000,
        "Honolulu": 500, "Kahului/Maui": 16, "Kona": 63, "Chicago": 5100,
        "Rockford": 172, "Fort Wayne": 200, "South Bend": 173, "Des Moines": 300,
        "Wichita": 290, "New Orleans": 550, "Baltimore": 1300, "Minneapolis": 1700,
        "Kansas City": 975, "St. Louis": 1200, "Omaha": 480, "Manchester": 100,
        "Albuquerque": 450, "New York": 11200, "Charlotte": 900, "Toledo": 290,
        "Wilmington": 150, "Portland": 1200, "Allentown": 420, "Pittsburgh": 1000,
        "San Juan": 1100, "Nashville": 650, "Austin": 975, "Dallas": 3300, 
        "Houston": 3300, "San Antonio": 1100, "Richmond": 600, "Seattle/Tacoma": 2000,
        "Spokane": 260
    }
}

# Helper function to flatten the nested dictionary
def flatten_centers(centers):
    return {city: demand for country, cities in centers.items() for city, demand in cities.items()}

flattened_centers = flatten_centers(centers)
print(f"Total Centers: {len(flattened_centers)}")

Total Centers: 65


In [2]:
DEFAULT_COST = 10000

def get_cost(src, dest, cost_data, default_cost=10000):
    return cost_data.get(src, {}).get(dest, default_cost)
#
#
cost = {
    # --- HUBS ---
    "CVG": {
        # Focus Cities
        "Leipzig": 1.5, "San Bernardino": 0.5,
        # Centers
        "Paris": 1.6, "Cologne": 1.5, "Hanover": 1.5, "Katowice": 1.4,
        "Barcelona": 1.5, "Madrid": 1.6, "Castle Donington": 1.4, "London": 1.6,
        "Mobile": 0.5, "Anchorage": 1.3, "Fairbanks": 1.4, "Phoenix": 0.5,
        "Los Angeles": 0.5, "Ontario": 0.5, "Riverside": 0.5, "Sacramento": 0.5,
        "San Francisco": 0.5, "Stockton": 0.5, "Denver": 0.5, "Hartford": 0.5,
        "Miami": 0.5, "Lakeland": 0.5, "Tampa": 0.5, "Atlanta": 0.5,
        "Chicago": 0.5, "Rockford": 0.5, "Fort Wayne": 0.5, "South Bend": 0.5,
        "Des Moines": 0.5, "Wichita": 0.5, "New Orleans": 0.5, "Baltimore": 0.5,
        "Minneapolis": 0.5, "Kansas City": 0.5, "St. Louis": 0.5, "Omaha": 0.5,
        "Manchester": 0.5, "Albuquerque": 0.5, "New York": 0.5, "Charlotte": 0.5,
        "Toledo": 0.5, "Wilmington": 0.5, "Portland": 0.5, "Allentown": 0.5,
        "Pittsburgh": 0.5, "San Juan": 0.5, "Nashville": 0.5, "Austin": 0.5,
        "Dallas": 0.5, "Houston": 0.5, "San Antonio": 0.5, "Richmond": 0.5,
        "Seattle/Tacoma": 0.5, "Spokane": 0.5,
        # N/A: Cagliari, Catania, Milan, Rome, Bengaluru, Coimbatore, Delhi, Mumbai,
        # Honolulu, Kahului/Maui, Kona, Hyderabad
    },
    "AFW": {
        # Focus Cities
        "San Bernardino": 0.5,
        # Centers
        "Mobile": 0.5, "Anchorage": 1, "Fairbanks": 1, "Phoenix": 0.5,
        "Los Angeles": 0.5, "Ontario": 0.5, "Riverside": 0.5, "Sacramento": 0.5,
        "San Francisco": 0.5, "Stockton": 0.5, "Denver": 0.5, "Hartford": 0.5,
        "Miami": 0.5, "Lakeland": 0.5, "Tampa": 0.5, "Atlanta": 0.5,
        "Honolulu": 0.5, "Kahului/Maui": 0.5, "Kona": 0.5, "Chicago": 0.5,
        "Rockford": 0.5, "Fort Wayne": 0.5, "South Bend": 0.5, "Des Moines": 0.5,
        "Wichita": 0.5, "New Orleans": 0.5, "Baltimore": 0.5, "Minneapolis": 0.5,
        "Kansas City": 0.5, "St. Louis": 0.5, "Omaha": 0.5, "Manchester": 0.5,
        "Albuquerque": 0.5, "New York": 0.5, "Charlotte": 0.5, "Toledo": 0.5,
        "Wilmington": 0.5, "Portland": 0.5, "Allentown": 0.5, "Pittsburgh": 0.5,
        "San Juan": 0.5, "Nashville": 0.5, "Austin": 0.25, "Houston": 0.25,
        "San Antonio": 0.25, "Richmond": 0.5, "Seattle/Tacoma": 0.5, "Spokane": 0.5
        # N/A: Dallas (listed as N/A in table for AFW), Leipzig, Hyderabad, All Europe, All India
    },
    
    # --- FOCUS CITIES ---
    "Leipzig": {
        # Focus Cities
        "Hyderabad": 1.6,
        # Centers
        "Paris": 0.5, "Cologne": 0.5, "Hanover": 0.5, "Bengaluru": 1.5,
        "Coimbatore": 1.5, "Delhi": 1.5, "Mumbai": 1.5, "Cagliari": 0.5,
        "Catania": 0.5, "Milan": 0.5, "Rome": 0.5, "Katowice": 0.5,
        "Barcelona": 0.5, "Madrid": 0.5, "Castle Donington": 0.5, "London": 0.75,
        "Hartford": 1.5, "Baltimore": 1.5, "Manchester": 1.5, "New York": 1.6,
        "Allentown": 1.5
    },
    "Hyderabad": {
        # Centers
        "Paris": 1.1, "Cologne": 1, "Hanover": 1, "Bengaluru": 0.5,
        "Coimbatore": 0.5, "Delhi": 0.5, "Mumbai": 0.5, "Cagliari": 1,
        "Catania": 1, "Milan": 1, "Rome": 1.1, "Katowice": 1,
        "Barcelona": 1, "Madrid": 1.1, "London": 1.1
    },
    "San Bernardino": {
        # Centers
        "Mobile": 0.5, "Anchorage": 0.7, "Fairbanks": 0.7, "Phoenix": 0.5,
        "Sacramento": 0.5, "San Francisco": 0.5, "Stockton": 0.5, "Denver": 0.5,
        "Hartford": 0.5, "Miami": 0.7, "Lakeland": 0.7, "Tampa": 0.7,
        "Atlanta": 0.6, "Honolulu": 0.5, "Kahului/Maui": 0.5, "Kona": 0.5,
        "Chicago": 0.5, "Rockford": 0.5, "Fort Wayne": 0.5, "South Bend": 0.5,
        "Des Moines": 0.5, "Wichita": 0.5, "New Orleans": 0.5, "Baltimore": 0.7,
        "Minneapolis": 0.5, "Kansas City": 0.5, "St. Louis": 0.5, "Omaha": 0.5,
        "Manchester": 0.7, "Albuquerque": 0.5, "New York": 0.7, "Charlotte": 0.7,
        "Toledo": 0.5, "Wilmington": 0.7, "Portland": 0.5, "Allentown": 0.7,
        "Pittsburgh": 0.6, "San Juan": 1, "Nashville": 0.5, "Austin": 0.5,
        "Dallas": 0.5, "Houston": 0.5, "San Antonio": 0.5, "Richmond": 0.7,
        "Seattle/Tacoma": 0.5, "Spokane": 0.5
        # N/A: Los Angeles, Ontario, Riverside
    }
}

In [3]:
#Optimization Model Setup
prob = LpProblem("Cargo_Network_Optimization", LpMinimize)

# --- VARIABLES ---
# Matches: x_ij = quantity sent from hub i to focus city j
x = LpVariable.dicts("x_Hub_to_Focus", 
                     (hubs.keys(), focus_cities.keys()), 
                     lowBound=0, cat='Continuous')

# Matches: y_ik = quantity sent from hub i to center k (DIRECT SHIPMENT)
y = LpVariable.dicts("y_Hub_to_Center", 
                     (hubs.keys(), flattened_centers.keys()), 
                     lowBound=0, cat='Continuous')

# Matches: z_jk = quantity sent from focus city j to center k
z = LpVariable.dicts("z_Focus_to_Center", 
                     (focus_cities.keys(), flattened_centers.keys()), 
                     lowBound=0, cat='Continuous')

# --- OBJECTIVE FUNCTION ---
# Minimize: sum(cost*x) + sum(cost*y) + sum(cost*z)
prob += (
    lpSum([get_cost(h, f, cost) * x[h][f] for h in hubs for f in focus_cities]) +
    lpSum([get_cost(h, c, cost) * y[h][c] for h in hubs for c in flattened_centers]) +
    lpSum([get_cost(f, c, cost) * z[f][c] for f in focus_cities for c in flattened_centers])
), "Total_Shipping_Cost"

# --- CONSTRAINTS ---

# 1. Hub Capacities
# Formula: sum(x_ij) + sum(y_ik) <= Capacity
for h in hubs:
    prob += (
        lpSum([x[h][f] for f in focus_cities]) + 
        lpSum([y[h][c] for c in flattened_centers])
    ) <= hubs[h]["capacity"], f"Hub_{h}_Capacity"

# 2. Focus City Capacity
# Formula: sum(x_ij) <= Capacity
for f in focus_cities:
    prob += lpSum([x[h][f] for h in hubs]) <= focus_cities[f]["capacity"], f"FocusCity_{f}_Capacity"

# 3. Flow Balance (Quantity Out = Quantity In)
# Formula: sum(z_jk) == sum(x_ij)
for f in focus_cities:
    prob += (
        lpSum([z[f][c] for c in flattened_centers]) == 
        lpSum([x[h][f] for h in hubs])
    ), f"Flow_Balance_{f}"

# 4. Center Demand Satisfaction
# Formula: sum(y_ik) + sum(z_jk) == Requirement
for c, demand in flattened_centers.items():
    prob += (
        lpSum([y[h][c] for h in hubs]) + 
        lpSum([z[f][c] for f in focus_cities])
    ) == demand, f"Demand_{c}"

print(f"Model built successfully with {len(prob.variables())} variables and {len(prob.constraints)} constraints.")
print("Decision variables: x = hub to focus, y = hub to center, z = focus to center.")

Model built successfully with 331 variables and 73 constraints.
Decision variables: x = hub to focus, y = hub to center, z = focus to center.


In [4]:
# Solve the optimization model and summarize the result.
solver = PULP_CBC_CMD(msg=False)
prob.solve(solver)

status = LpStatus[prob.status]
objective_value = prob.objective.value()

print(f"Solver Status: {status}")
if status != "Optimal":
    raise RuntimeError(f"Expected an optimal solution, got {status}.")

print(f"Total Cost: ${objective_value:,.2f}")

total_demand = sum(flattened_centers.values())
total_direct = sum((y[h][c].varValue or 0) for h in hubs for c in flattened_centers)
total_via_focus = sum((z[f][c].varValue or 0) for f in focus_cities for c in flattened_centers)
total_shipped = total_direct + total_via_focus

summary_df = pd.DataFrame([
    {"metric": "Total demand", "tons": total_demand},
    {"metric": "Direct from hubs", "tons": total_direct},
    {"metric": "Via focus cities", "tons": total_via_focus},
    {"metric": "Total shipped", "tons": total_shipped},
])
summary_df["tons"] = summary_df["tons"].map(lambda value: f"{value:,.1f}")
display(summary_df)


Solver Status: Optimal
Total Cost: $200,863.75


,metric,tons
0,Total demand,"133,747.0"
1,Direct from hubs,"87,502.0"
2,Via focus cities,"46,245.0"
3,Total shipped,"133,747.0"


In [5]:
# Show the non-zero shipping plan.
shipments = []

for h in hubs:
    for f in focus_cities:
        tons = x[h][f].varValue or 0
        if tons > 1e-6:
            unit_cost = get_cost(h, f, cost)
            shipments.append({
                "lane_type": "hub_to_focus",
                "source": h,
                "destination": f,
                "tons": tons,
                "unit_cost": unit_cost,
                "shipment_cost": tons * unit_cost,
            })

for h in hubs:
    for c in flattened_centers:
        tons = y[h][c].varValue or 0
        if tons > 1e-6:
            unit_cost = get_cost(h, c, cost)
            shipments.append({
                "lane_type": "hub_to_center",
                "source": h,
                "destination": c,
                "tons": tons,
                "unit_cost": unit_cost,
                "shipment_cost": tons * unit_cost,
            })

for f in focus_cities:
    for c in flattened_centers:
        tons = z[f][c].varValue or 0
        if tons > 1e-6:
            unit_cost = get_cost(f, c, cost)
            shipments.append({
                "lane_type": "focus_to_center",
                "source": f,
                "destination": c,
                "tons": tons,
                "unit_cost": unit_cost,
                "shipment_cost": tons * unit_cost,
            })

shipments_df = pd.DataFrame(shipments)
shipments_df = shipments_df.sort_values(["lane_type", "source", "destination"]).reset_index(drop=True)

print(f"Non-zero lanes: {len(shipments_df)}")
print(f"Selected lanes using penalty cost: {(shipments_df['unit_cost'] >= DEFAULT_COST).sum()}")

formatted_shipments = shipments_df.copy()
formatted_shipments["tons"] = formatted_shipments["tons"].map(lambda value: f"{value:,.1f}")
formatted_shipments["unit_cost"] = formatted_shipments["unit_cost"].map(lambda value: f"${value:,.2f}")
formatted_shipments["shipment_cost"] = formatted_shipments["shipment_cost"].map(lambda value: f"${value:,.2f}")
display(formatted_shipments)

if (shipments_df["unit_cost"] >= DEFAULT_COST).any():
    raise AssertionError("The optimized route selected a penalty-cost lane.")


Non-zero lanes: 67
Selected lanes using penalty cost: 0


,lane_type,source,destination,tons,unit_cost,shipment_cost
0,focus_to_center,Leipzig,Bengaluru,"9,100.0",$1.50,"$13,650.00"
1,focus_to_center,Leipzig,Cagliari,90.0,$0.50,$45.00
2,focus_to_center,Leipzig,Catania,185.0,$0.50,$92.50
3,focus_to_center,Leipzig,Coimbatore,570.0,$1.50,$855.00
4,focus_to_center,Leipzig,Delhi,"19,000.0",$1.50,"$28,500.00"
...,...,...,...,...,...,...
62,hub_to_center,CVG,Spokane,260.0,$0.50,$130.00
63,hub_to_center,CVG,Stockton,240.0,$0.50,$120.00
64,hub_to_center,CVG,Tampa,"1,600.0",$0.50,$800.00
65,hub_to_center,CVG,Toledo,290.0,$0.50,$145.00


In [6]:
# Verify capacity, flow-balance, and demand-satisfaction constraints.
TOL = 1e-5
check_rows = []

for h in hubs:
    sent_to_focus = sum((x[h][f].varValue or 0) for f in focus_cities)
    sent_direct = sum((y[h][c].varValue or 0) for c in flattened_centers)
    used = sent_to_focus + sent_direct
    cap = hubs[h]["capacity"]
    check_rows.append({
        "constraint": "Hub capacity",
        "entity": h,
        "used_or_inbound": used,
        "limit_or_outbound": cap,
        "status": "OK" if used <= cap + TOL else "VIOLATED",
    })

for f in focus_cities:
    inbound = sum((x[h][f].varValue or 0) for h in hubs)
    cap = focus_cities[f]["capacity"]
    check_rows.append({
        "constraint": "Focus city capacity",
        "entity": f,
        "used_or_inbound": inbound,
        "limit_or_outbound": cap,
        "status": "OK" if inbound <= cap + TOL else "VIOLATED",
    })

for f in focus_cities:
    inbound = sum((x[h][f].varValue or 0) for h in hubs)
    outbound = sum((z[f][c].varValue or 0) for c in flattened_centers)
    check_rows.append({
        "constraint": "Focus city flow balance",
        "entity": f,
        "used_or_inbound": inbound,
        "limit_or_outbound": outbound,
        "status": "OK" if abs(inbound - outbound) <= TOL else "VIOLATED",
    })

check_df = pd.DataFrame(check_rows)
formatted_checks = check_df.copy()
for col in ["used_or_inbound", "limit_or_outbound"]:
    formatted_checks[col] = formatted_checks[col].map(lambda value: f"{value:,.1f}")
display(formatted_checks)

demand_mismatches = []
for center, demand in flattened_centers.items():
    received_direct = sum((y[h][center].varValue or 0) for h in hubs)
    received_from_focus = sum((z[f][center].varValue or 0) for f in focus_cities)
    total_received = received_direct + received_from_focus
    if abs(total_received - demand) > 0.01:
        demand_mismatches.append((center, demand, total_received))

print(f"Demand centers checked: {len(flattened_centers)}")
if demand_mismatches:
    for center, demand, received in demand_mismatches:
        print(f"{center}: received {received:.1f}, required {demand:.1f}")
    raise AssertionError("One or more demand centers were not satisfied.")

print("All demand centers: requirements met exactly (OK)")

if (check_df["status"] != "OK").any():
    raise AssertionError("One or more capacity or flow-balance checks failed.")

expected_objective = 200_863.75
if abs(objective_value - expected_objective) > 0.01:
    raise AssertionError(f"Expected objective ${expected_objective:,.2f}, got ${objective_value:,.2f}.")

print(f"Objective matches report: ${expected_objective:,.2f} (OK)")


,constraint,entity,used_or_inbound,limit_or_outbound,status
0,Hub capacity,CVG,"95,650.0","95,650.0",OK
1,Hub capacity,AFW,"38,097.0","44,350.0",OK
2,Focus city capacity,Leipzig,"46,245.0","85,000.0",OK
3,Focus city capacity,Hyderabad,0.0,"19,000.0",OK
4,Focus city capacity,San Bernardino,0.0,"36,000.0",OK
5,Focus city flow balance,Leipzig,"46,245.0","46,245.0",OK
6,Focus city flow balance,Hyderabad,0.0,0.0,OK
7,Focus city flow balance,San Bernardino,0.0,0.0,OK


Demand centers checked: 65
All demand centers: requirements met exactly (OK)
Objective matches report: $200,863.75 (OK)
